# Midstream result 2023

In [2]:
import pandas as pd

In [3]:
# contribution data
path = "/Users/spencerzhang/GitHub/PhD/North-America-Gas-2021/revision_data/leontief/"

prod_contribution = pd.read_csv(path + "row_contribution_percentage_production_long_gt1_scaled100.csv")
rnd_contribution = pd.read_csv(path + "column_contribution_percentage_receiving_long_gt1_scaled100.csv")

In [4]:
# production data
path = '/Users/spencerzhang/GitHub/PhD/North-America-Gas-2021/revision_data/state production consumption/state_production_2023_with_ids.csv'
production_data = pd.read_csv(path)

In [ ]:
import pandas as pd

path_data = '/Users/spencerzhang/GitHub/PhD/North-America-Gas-2021/revision_data/state production consumption/'
path_leon = "/Users/spencerzhang/GitHub/PhD/North-America-Gas-2021/revision_data/leontief/"
# File paths
row_contrib_fp = path_leon+"row_contribution_percentage_production_long_gt1_scaled100.csv"
state_prod_fp = path_data+"state_production_2023_with_ids.csv"

# Read files
row_contrib_df = pd.read_csv(row_contrib_fp)
state_prod_df = pd.read_csv(state_prod_fp)

# Remove rows where source == 'US'
row_contrib_df = row_contrib_df[row_contrib_df['source'] != 'US']

# Merge: 'source' in row_contrib_df to 'state' in state_prod_df
merged_df = pd.merge(row_contrib_df, state_prod_df, left_on='source', right_on='state', how='left')

# Calculate transported gas
merged_df['transported_gas'] = (
    merged_df['natural_gas_dry_production_MMcf'] *
    merged_df['contribution_percentage'] * 0.01
)

# Save result
merged_df.to_csv(path_leon + "merged_transported_gas.csv", index=False)

print("Saved merged_transported_gas_from_production.csv")


Saved merged_transported_gas.csv


In [8]:

# File paths
col_contrib_fp = path_leon+"column_contribution_percentage_receiving_long_gt1_scaled100.csv"
state_cons_fp = path_data+"state_consumption_2023_with_ids.csv"

# Read files
col_contrib_df = pd.read_csv(col_contrib_fp)
state_cons_df = pd.read_csv(state_cons_fp)

# Remove rows where source == 'US'
col_contrib_df = col_contrib_df[col_contrib_df['source'] != 'US']

# Merge: 'destination' in col_contrib_df to 'state' in state_cons_df
merged_df = pd.merge(col_contrib_df, state_cons_df, left_on='destination', right_on='state', how='left')

# Calculate transported gas
merged_df['transported_gas'] = (
    merged_df['natural_gas_consumption_value_MMcf'] *
    merged_df['contribution_percentage'] * 0.01
)

# Save result
merged_df.to_csv(path_leon+"merged_transported_gas_consumption.csv", index=False)

print("Saved merged_transported_gas_consumption.csv")


Saved merged_transported_gas_consumption.csv


In [ ]:
# deals with missing distance
path_mid = "/Users/spencerzhang/GitHub/PhD/North-America-Gas-2021/revision_data/midstream_state_level/"
dist_df  = pd.read_csv(path_mid+ "processing_to_delivery_distance_long_km.csv")     # processing_id, delivery_id, distance_km
flows_df = pd.read_csv(path_mid+ "merged_transported_gas_consumption.csv")          # source, destination, ...

merged = flows_df.merge(
    dist_df[['processing_id', 'delivery_id', 'distance_km']],
    left_on=['source', 'destination'],
    right_on=['processing_id', 'delivery_id'],
    how='left'  # keep all flows; attach distance when available
).drop(columns=['processing_id', 'delivery_id'])

# Identify labels
texas_label = "Texas"
louisiana_label = "Louisiana"

# Mean distance from TX & LA to each destination
mean_dist_by_dest = (
    dist_df[dist_df["processing_id"].isin([texas_label, louisiana_label])]
    .groupby("delivery_id")["distance_km"].mean()
)

# Apply only to rows where source is Federal Offshore--Gulf of America
mask = merged["source"] == "Federal Offshore--Gulf of America"
merged.loc[mask, "distance_km"] = merged.loc[mask, "destination"].map(mean_dist_by_dest)

# 75th percentile of all distances (km)
p75_km = dist_df["distance_km"].quantile(0.75)

# case-insensitive match for "International"
mask = merged["source"].astype(str).str.strip().str.lower().eq("international") | \
       merged["destination"].astype(str).str.strip().str.lower().eq("international")

merged.loc[mask, "distance_km"] = p75_km

merged.to_csv(path_mid+"flows_with_distances.csv", index=False)


In [15]:
path_mid = "/Users/spencerzhang/GitHub/PhD/North-America-Gas-2021/revision_data/midstream_state_level/"
midstream_df = pd.read_csv(path_mid + "flows_with_distances_filled.csv")

In [17]:
# Constants (match your R code)
midstream_emission_factor = 4.00        # kgCO2e/MMCF-km
midstream_emission_factor_low = 3.77
midstream_emission_factor_high = 4.28
tortuosity_factor_low = 1.03
tortuosity_factor_high = 1.16
tortuosity_factor = (tortuosity_factor_low + tortuosity_factor_high) / 2.0
MJ_per_MMCF = 1094000                # MJ/MMCF
kgCO2_mmcf_to_g_MJ = 1000.0 / MJ_per_MMCF  # kg/mmscf -> g/MJ

# Read input
df = pd.read_csv(path_mid+"flows_with_distances_filled.csv")

# Compute EFs (gCO2e/MJ)
km = df['distance_km'] 
df['midstream_EF_g_MJ'] = km * midstream_emission_factor * kgCO2_mmcf_to_g_MJ * tortuosity_factor
df['midstream_EF_g_MJ_low'] = km * midstream_emission_factor_low * kgCO2_mmcf_to_g_MJ * tortuosity_factor_low
df['midstream_EF_g_MJ_high'] = km * midstream_emission_factor_high * kgCO2_mmcf_to_g_MJ * tortuosity_factor_high

# Save
df.to_csv(path_mid+"flows_with_midstream_ef.csv", index=False)


In [35]:
path_data = '/Users/spencerzhang/GitHub/PhD/North-America-Gas-2021/revision_data/state production consumption/'
prod_path = path_data+"state_production_2023_with_ids.csv"
cons_path = path_data+"state_consumption_2023_with_ids.csv"

prod_df = pd.read_csv(prod_path)
cons_df = pd.read_csv(cons_path)

prod_col = 'natural_gas_dry_production_MMcf'
cons_col = 'natural_gas_consumption_value_MMcf'

# Make sure the column exists and is numeric
prod_df[prod_col] = pd.to_numeric(prod_df[prod_col], errors='coerce')
prod_df = prod_df[prod_df['state'] != "U.S."]
prod_df = prod_df[prod_df['state'] != "International"]
# If there can be multiple rows per state, aggregate first
prod_by_state = (
    prod_df.groupby('state', as_index=False)[prod_col]
           .sum()
)
# Sort and take top 10
top10_prod = prod_by_state.sort_values(prod_col, ascending=False).head(10)
# (Optional) get just the list of state names, in order
top10_prod = top10_prod['state'].tolist()

# Make sure the column exists and is numeric
cons_df[cons_col] = pd.to_numeric(cons_df[cons_col], errors='coerce')
cons_df = cons_df[cons_df['state'] != "U.S."]
cons_df = cons_df[cons_df['state'] != "International"]
# If there can be multiple rows per state, aggregate first
cons_by_state = (
    cons_df.groupby('state', as_index=False)[cons_col]
           .sum()
)
# Sort and take top 10
top10_cons = cons_by_state.sort_values(cons_col, ascending=False).head(10)
# (Optional) get just the list of state names, in order
top10_cons = top10_cons['state'].tolist()

print(top10_prod)
print(top10_cons)

['Texas', 'Pennsylvania', 'Louisiana', 'West Virginia', 'Oklahoma', 'New Mexico', 'Ohio', 'Colorado', 'Wyoming', 'North Dakota']
['Texas', 'California', 'Louisiana', 'Pennsylvania', 'Florida', 'New York', 'Ohio', 'Illinois', 'Michigan', 'Indiana']


In [80]:
import pandas as pd
import plotly.graph_objects as go
import numpy as np
import matplotlib.cm as cm
import matplotlib.colors as mcolors

# --- Load & filter (adjust as you like) ---
df = pd.read_csv(path_mid + "flows_with_midstream_ef.csv")
df = df[df['source'].isin(top10_prod) & df['destination'].isin(top10_cons)]
df = df[~((df['source'] == 'International') & (df['destination'] == 'International'))]

#df = df.nlargest(30, "transported_gas").copy()   # or 10, or all

# Labels for two distinct node sets
df["source_label"] = df["source"].astype(str) + " prod"
df["dest_label"]   = df["destination"].astype(str) + " cons"

# Order nodes by total volume (nice layout)
prod_order = (df.groupby("source_label")["transported_gas"]
                .sum().sort_values(ascending=False).index.tolist())
cons_order = (df.groupby("dest_label")["transported_gas"]
                .sum().sort_values(ascending=False).index.tolist())

nodes = prod_order + cons_order
node_id = {name: i for i, name in enumerate(nodes)}

# Fix positions: prod at x=0, cons at x=1 (keeps two distinct columns)
x_prod = [0.1] * len(prod_order)
x_cons = [1.0] * len(cons_order)
y_prod = np.linspace(0.02, 0.98, len(prod_order))  # spread vertically
y_cons = np.linspace(0.02, 0.98, len(cons_order))
node_x = x_prod + x_cons
node_y = y_prod.tolist() + y_cons.tolist()

# Color links by midstream CI
norm  = mcolors.Normalize(vmin=df["midstream_EF_g_MJ"].min(),
                          vmax=df["midstream_EF_g_MJ"].max())
cmap  = cm.get_cmap("RdYlGn_r")  # high CI = red, low CI = green
link_colors = [mcolors.to_hex(cmap(norm(ci))) for ci in df["midstream_EF_g_MJ"]]

import re

# ... your existing code up to nodes/node_id ...

# Labels shown on the graph (strip " prod"/" cons" for display only)
nodes_display = [re.sub(r"\s+(prod|cons)\b", "", n) for n in nodes]

fig = go.Figure(go.Sankey(
    arrangement="snap",
    node=dict(
        pad=14,
        thickness=16,
        label=nodes_display,    # <- display without suffixes
        color="lightgrey",
        x=node_x,
        y=node_y
    ),
    link=dict(
        source=df["source_label"].map(node_id),
        target=df["dest_label"].map(node_id),
        value=df["transported_gas"],
        color=link_colors,
        hovertemplate=(
            "From %{source.label} → %{target.label}<br>" +
            "Transported gas: %{value:,}<br>" +
            "Midstream CI (g/MJ): %{customdata:.3f}<extra></extra>"
        ),
        customdata=df["midstream_EF_g_MJ"]
    )
))


fig.update_layout(
    font=dict(family="Helvetica, Arial, sans-serif", size=12),
    hoverlabel=dict(font_family="Helvetica, Arial, sans-serif")
)

# # ---------- 1) Color legend (continuous colorbar for midstream CI) ----------
ci_min = float(df["midstream_EF_g_MJ"].min())
ci_max = float(df["midstream_EF_g_MJ"].max())

fig.add_trace(
    go.Scatter(
        x=[None], y=[None], mode="markers",
        marker=dict(
            colorscale="RdYlGn_r",
            cmin=ci_min, cmax=ci_max,
            color=[ci_min, ci_max],
            size=1,
            showscale=True,
            colorbar=dict(
                title="Midstream CI (g CO₂e/MJ)",
                thickness=26,            # <- width of the colorbar (px)
                thicknessmode="pixels",  # explicit; default is pixels
                len=0.8, lenmode="fraction",
                y=0.5, x=1.03,           # position
                bgcolor="white",
                outlinewidth=0,
                ticks="outside",
                tickfont=dict(size=11),
                titlefont=dict(size=12)
            )
        ),
        hoverinfo="none", showlegend=False
    )
)

fig.update_layout(
    template="plotly_white",        # or "none"
    paper_bgcolor="white",
    plot_bgcolor="white"
)

# Hide any axes created by the dummy scatter (remove grid, ticks, and axis lines)
fig.update_xaxes(visible=False, showgrid=False, zeroline=False, showline=False)
fig.update_yaxes(visible=False, showgrid=False, zeroline=False, showline=False)

# (Optional) tighten margins so legends fit nicely
fig.update_layout(margin=dict(l=10, r=100, t=60, b=10))

import re

# 1) Hide built-in node labels
fig.data[0].node.label = [""] * len(nodes)



fig.show()

# Vector export (crisp at any zoom)
fig.write_image("production_to_consumption_sankey_no_anno.svg", format="svg", width=800, height=400)


# volume weighted average midstream CI

In [82]:
# ---- Config ----
path = "/Users/spencerzhang/GitHub/PhD/North-America-Gas-2021/revision_data/midstream_state_level/"
csv_path = path + "flows_with_midstream_ef.csv"  # change to your path if needed
weight_col = "transported_gas"
value_cols: list[str] = [
    "distance_km",
    "midstream_EF_g_MJ",
    "midstream_EF_g_MJ_low",
    "midstream_EF_g_MJ_high",
]

# ---- Load ----
df = pd.read_csv(csv_path)

# Ensure numeric types (coerce errors to NaN)
df[weight_col] = pd.to_numeric(df[weight_col], errors="coerce")
for c in value_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

def weighted_avg(g: pd.DataFrame, col: str, wcol: str = weight_col) -> float:
    """Volume-weighted average of `col` using `wcol` as weights, NaN-safe."""
    v = g[col]
    w = g[wcol]
    m = v.notna() & w.notna()
    if not m.any():
        return float("nan")
    w_sum = w[m].sum()
    if w_sum == 0:
        return float("nan")
    return float((v[m] * w[m]).sum() / w_sum)

def summarize(group_key: str) -> pd.DataFrame:
    """Group by group_key and return weighted averages + total transported_gas."""
    grp = df.groupby(group_key, dropna=False)
    out = grp.apply(
        lambda g: pd.Series(
            {
                "distance_km_weighted":            weighted_avg(g, "distance_km"),
                "midstream_EF_g_MJ_weighted":      weighted_avg(g, "midstream_EF_g_MJ"),
                "midstream_EF_g_MJ_low_weighted":  weighted_avg(g, "midstream_EF_g_MJ_low"),
                "midstream_EF_g_MJ_high_weighted": weighted_avg(g, "midstream_EF_g_MJ_high"),
                "total_transported_gas":           g[weight_col].sum(skipna=True),
            }
        )
    ).reset_index()
    return out

# ---- Results ----
by_source = summarize("source")
by_destination = summarize("destination")

# Optional: sort by total transported gas (descending)
by_source = by_source.sort_values("total_transported_gas", ascending=False)
by_destination = by_destination.sort_values("total_transported_gas", ascending=False)

# Optional: save
by_source.to_csv(path + "midstream_CI_weighted_by_source.csv", index=False)
by_destination.to_csv(path+"midstream_CI_weighted_by_destination.csv", index=False)

# Quick peek
print(by_source.head(10))
print(by_destination.head(10))


           source  distance_km_weighted  midstream_EF_g_MJ_weighted  \
17          Texas           1158.552832                    4.638447   
16   Pennsylvania            630.653870                    2.524921   
9       Louisiana           1120.927341                    4.487808   
6   International           2478.890293                    9.924625   
15       Oklahoma            732.599464                    2.933076   
20  West Virginia            717.635732                    2.873167   
14           Ohio            874.365318                    3.500658   
12     New Mexico           1070.866041                    4.287380   
4        Colorado            676.455804                    2.708297   
21        Wyoming            856.949203                    3.430930   

    midstream_EF_g_MJ_low_weighted  midstream_EF_g_MJ_high_weighted  \
17                        4.112227                         5.257754   
16                        2.238475                         2.862039   
9    

# abstract figure

In [129]:
import pandas as pd
import numpy as np
from pathlib import Path
import plotly.graph_objects as go
from plotly.colors import sample_colorscale

# ---------- Load data ----------
prod_path = Path("/Users/spencerzhang/GitHub/PhD/North-America-Gas-2021/revision_data/figure/plot_df/abstract_prod.csv")
flow_path = Path("/Users/spencerzhang/GitHub/PhD/North-America-Gas-2021/revision_data/midstream_state_level/flows_with_midstream_ef.csv")

prod_df = pd.read_csv(prod_path)
flow_df = pd.read_csv(flow_path)

# Required columns
req_prod = ["Basin", "prod_state", "Gas_mmcf", "CI"]
req_flow = ["source", "destination", "transported_gas", "midstream_EF_g_MJ"]
for col in req_prod:
    if col not in prod_df.columns:
        raise ValueError(f"Missing column '{col}' in abstract_prod.csv")
for col in req_flow:
    if col not in flow_df.columns:
        raise ValueError(f"Missing column '{col}' in flows_with_midstream_ef.csv")

# Coerce numerics
prod_df["Gas_mmcf"] = pd.to_numeric(prod_df["Gas_mmcf"], errors="coerce").fillna(0.0)
prod_df["CI"] = pd.to_numeric(prod_df["CI"], errors="coerce")
flow_df["transported_gas"] = pd.to_numeric(flow_df["transported_gas"], errors="coerce").fillna(0.0)
flow_df["midstream_EF_g_MJ"] = pd.to_numeric(flow_df["midstream_EF_g_MJ"], errors="coerce")

# Align naming
flow_df = flow_df.rename(columns={"source": "prod_state"})

# (Optional) ensure unique pairs; comment out if you want to preserve duplicates
prod_df = prod_df.drop_duplicates(subset=["Basin", "prod_state"])
prod_df['Basin'] = prod_df['Basin'].str.title()
prod_df['Basin'] = prod_df['Basin'].replace('Gom', 'Gulf of Mexico')
prod_df['Basin'] = prod_df['Basin'].replace('Ft Worth Basin', 'Fort Worth Basin')

flow_df = flow_df.drop_duplicates(subset=["prod_state", "destination"])

# ---------------- 2) Filters: top 15 basins & top 10 destinations ----------------
top15_basins = (
    prod_df.groupby("Basin")["Gas_mmcf"]
           .sum()
           .sort_values(ascending=False)
           .head(15)
           .index
           .tolist()
)
prod_df = prod_df[prod_df["Basin"].isin(top15_basins)]

top10_dests = (
    flow_df.groupby("destination")["transported_gas"]
           .sum()
           .sort_values(ascending=False)
           .head(13)
           .index
           .tolist()
)
flow_df = flow_df[flow_df["destination"].isin(top10_dests)]

# Keep only middle nodes present in both after filtering
valid_states = set(prod_df["prod_state"]).intersection(set(flow_df["prod_state"]))
prod_df = prod_df[prod_df["prod_state"].isin(valid_states)]
flow_df = flow_df[flow_df["prod_state"].isin(valid_states)]

if prod_df.empty or flow_df.empty:
    raise ValueError("After filtering, one or both datasets are empty. Relax thresholds or check input files.")

# ---------------- 3) Match inflow/outflow at middle nodes by scaling ----------------
in_totals = prod_df.groupby("prod_state")["Gas_mmcf"].sum()
out_totals = flow_df.groupby("prod_state")["transported_gas"].sum()

# For each prod_state, scale both sides to the smaller of in vs out
scale_in = {}
scale_out = {}
for state in sorted(valid_states):
    inflow = float(in_totals.get(state, 0.0))
    outflow = float(out_totals.get(state, 0.0))
    if inflow <= 0 or outflow <= 0:
        # If one side is zero, leave values as-is (they'll be zero anyway)
        scale_in[state] = 1.0
        scale_out[state] = 1.0
    else:
        target = min(inflow, outflow)
        scale_in[state] = target / inflow
        scale_out[state] = target / outflow

prod_df = prod_df.assign(Gas_mmcf_scaled=prod_df.apply(lambda r: r["Gas_mmcf"] * scale_in[r["prod_state"]], axis=1))
flow_df = flow_df.assign(transported_gas_scaled=flow_df.apply(lambda r: r["transported_gas"] * scale_out[r["prod_state"]], axis=1))

# ---------------- 4) Order nodes by total Gas_mmcf (right uses scaled transported_gas) ----------------
basin_order = (
    prod_df.groupby("Basin")["Gas_mmcf_scaled"]
           .sum()
           .sort_values(ascending=False)
           .index
           .tolist()
)
middle_order = (
    prod_df.groupby("prod_state")["Gas_mmcf_scaled"]
           .sum()
           .sort_values(ascending=False)
           .index
           .tolist()
)
dest_order = (
    flow_df.groupby("destination")["transported_gas_scaled"]
           .sum()
           .sort_values(ascending=False)
           .index
           .tolist()
)

labels = basin_order + middle_order + dest_order
idx_basin  = {b: i for i, b in enumerate(basin_order)}
idx_middle = {m: i + len(basin_order) for i, m in enumerate(middle_order)}
idx_dest   = {d: i + len(basin_order) + len(middle_order) for i, d in enumerate(dest_order)}

# ---------------- 5) Discrete color helpers (SEPARATE scales for bs and sd) ----------------
def assign_discrete_colors(values, bin_edges, bin_colors):
    vals = np.asarray(values, dtype=float)
    # digitize to 0..n_bins-1
    idx = np.digitize(vals, bin_edges, right=True)
    idx = np.clip(idx, 0, len(bin_colors) - 1)
    return [bin_colors[i] for i in idx]

def build_legend_labels(all_vals, bin_edges):
    labels = []
    lo = float(np.nanmin(all_vals))
    for edge in bin_edges:
        labels.append(f"{lo:.2f} – {edge:.2f}")
        lo = float(edge)
    labels.append(f"{lo:.2f} – {float(np.nanmax(all_vals)):.2f}")
    return labels

# --- Basin→State (bs) scale from CI only ---
bs_metric_series = prod_df["CI"].astype(float)
bs_all = bs_metric_series.values
# 9 edges → 10 bins; you can replace with fixed thresholds if you prefer
bs_bin_edges = np.nanquantile(bs_all, [i/10 for i in range(1, 10)])
# 10-color palette for bs (greens→reds)
bs_bin_colors = [
    "#006837", "#1a9850", "#66bd63", "#a6d96a", "#d9ef8b",
    "#fee08b", "#fdae61", "#f46d43", "#d73027", "#a50026"
]
bs_colors = assign_discrete_colors(bs_metric_series.tolist(), bs_bin_edges, bs_bin_colors)
bs_legend_labels = build_legend_labels(bs_all, bs_bin_edges)  # if you want a legend

# --- State→Destination (sd) scale from midstream_EF_g_MJ only ---
sd_metric_series = flow_df["midstream_EF_g_MJ"].astype(float)
sd_all = sd_metric_series.values
sd_bin_edges = np.nanquantile(sd_all, [i/8 for i in range(1, 8)])
# 10-color palette for sd (choose a distinct scheme to visually separate scales)
sd_bin_colors = [
    "#006837", "#1a9850", "#66bd63", "#a6d96a", "#d9ef8b",
    "#fee08b", "#fdae61", "#f46d43"
]
sd_colors = assign_discrete_colors(sd_metric_series.tolist(), sd_bin_edges, sd_bin_colors)
sd_legend_labels = build_legend_labels(sd_all, sd_bin_edges)  # if you want a legend


# ---------------- 6) Build Sankey link arrays ----------------
# Left → Middle
bs_sources = prod_df["Basin"].map(idx_basin).tolist()
bs_targets = prod_df["prod_state"].map(idx_middle).tolist()
bs_values  = prod_df["Gas_mmcf_scaled"].astype(float).tolist()
#bs_colors  = to_colors(prod_df["CI"].tolist(), "RdYlGn_r")  # CI colors

# Middle → Right
sd_sources = flow_df["prod_state"].map(idx_middle).tolist()
sd_targets = flow_df["destination"].map(idx_dest).tolist()
sd_values  = flow_df["transported_gas_scaled"].astype(float).tolist()
#sd_colors  = to_colors(flow_df["midstream_EF_g_MJ"].tolist(), "RdYlGn_r")  # midstream_EF colors

sources_idx = bs_sources + sd_sources
targets_idx = bs_targets + sd_targets
values_flow = bs_values  + sd_values
link_colors = bs_colors  + sd_colors

# Hover data
bs_custom = np.column_stack([
    prod_df["Basin"].astype(str).values,
    prod_df["prod_state"].astype(str).values,
    np.asarray(bs_values, dtype=float),
    np.asarray(prod_df["CI"], dtype=float)
])
sd_custom = np.column_stack([
    flow_df["prod_state"].astype(str).values,
    flow_df["destination"].astype(str).values,
    np.asarray(sd_values, dtype=float),
    np.asarray(flow_df["midstream_EF_g_MJ"], dtype=float)
])
customdata = np.vstack([bs_custom, sd_custom])

hovertemplate = (
    "<b>%{customdata[0]}</b> → <b>%{customdata[1]}</b><br>"
    "Flow (MMcf): %{customdata[2]:,.2f}<br>"
    "Metric: %{customdata[3]:,.4g}<extra></extra>"
)

# ---------------- 7) Plot ----------------
fig = go.Figure(
    go.Sankey(
        arrangement="snap",
        node=dict(
            label=labels,
            pad=16,
            thickness=18,
            line=dict(width=0.5, color="rgba(0,0,0,0.3)")
        ),
        link=dict(
            source=sources_idx,
            target=targets_idx,
            value=values_flow,
            color=link_colors,
            customdata=customdata,
            hovertemplate=hovertemplate
        ),
    )
)

fig.update_layout(
    #title="Basin → prod_state → destination (Top 15 basins, Top 10 destinations; flows matched at middle)",
    font=dict(
        family="Helvetica",
        size=12
    ),
    font_size=12,
    width=1200,
    height=720,
    margin=dict(l=10, r=10, t=60, b=10)
)

# Save as high-quality SVG
output_path = "/Users/spencerzhang/GitHub/PhD/North-America-Gas-2021/revision_data/figure/abstract_sankey_diagram.svg"
fig.write_image(output_path, format="svg")
print(f"Saved SVG to {output_path}")


fig.show()


Saved SVG to /Users/spencerzhang/GitHub/PhD/North-America-Gas-2021/revision_data/figure/abstract_sankey_diagram.svg


In [131]:
import plotly.graph_objects as go

def make_discrete_legend(bin_colors, labels, title=""):
    """
    Create a horizontal stacked-bar legend figure for discrete bins.
    - bin_colors: list[str] of hex colors (length N)
    - labels: list[str] of value-range labels (length N)
    """
    fig = go.Figure()

    # Draw a horizontal color strip as N stacked bars of equal width
    for color, label in zip(bin_colors, labels):
        fig.add_trace(go.Bar(
            x=[1], y=[1],
            marker=dict(color=color),
            name=label,
            orientation="h",
            showlegend=True,   # keeps a standard legend with labels too
            hoverinfo="none"
        ))

    fig.update_layout(
        barmode='stack',
        title=title,
        # compact horizontal legend with labels
           font=dict(
        family="Helvetica",
        size=12
    ),
        legend=dict(
            orientation="h",
            yanchor="bottom", y=1.02,
            xanchor="center", x=0.5,
            title=None,
            font=dict(size=11)
        ),
        # hide axes for a clean legend-only look
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        margin=dict(l=10, r=10, t=40, b=10),
        height=120
    )
    return fig

# Build the two legend figures
ci_legend_fig = make_discrete_legend(bs_bin_colors, bs_legend_labels)
ef_legend_fig = make_discrete_legend(sd_bin_colors, sd_legend_labels)

# Show them (separate windows/tabs depending on your environment)
ci_legend_fig.show()
ef_legend_fig.show()

# Optionally save to HTML
# ci_legend_fig.write_html("ci_legend.html")
# ef_legend_fig.write_html("ef_legend.html")


# Save legends as high-quality SVG
ci_legend_fig.write_image("/Users/spencerzhang/GitHub/PhD/North-America-Gas-2021/revision_data/figure/ci_legend.svg", format="svg")
ef_legend_fig.write_image("/Users/spencerzhang/GitHub/PhD/North-America-Gas-2021/revision_data/figure/ef_legend.svg", format="svg")

print("Saved ci_legend.svg and ef_legend.svg")


Saved ci_legend.svg and ef_legend.svg
